## SST

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import torch

import sys
from pathlib import Path
import os

from scipy.io import loadmat

# Add parent directory to path to access src module
sys.path.insert(0, str(Path.cwd().parent.parent))
DATA_DIR = Path.cwd() / "datasets"  

import helpers
from importlib import reload
helpers = reload(helpers)
from helpers import eval_all, plot_training_curves, evaluate_model, plot_reconstruction, plot_sst_with_buoys, plot_sst_overview_and_availability

from src.datamanager import DataManager
from src.engine import SHREDEngine
from src.train import fit, evaluate
from src.mSHRED.mSHRED import mSHRED

# get device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Exploration

In [29]:
# Load SST data -> X RAW
X_raw = np.load(DATA_DIR / "sst_all_ROM.npy")
print(f"Loaded SST data with shape: {X_raw.shape}")
S_raw = np.load(DATA_DIR / "buoy_all_ROM.npy")
print(f"Loaded Buoy data with shape: {S_raw.shape}\n")

loc = pd.read_csv(DATA_DIR / "stations_location.csv", dtype={"station": str})
S_names = loc["station"].to_numpy()
S_lat = loc["lat"].to_numpy()
S_lon = loc["lon"].to_numpy()

for i, name in enumerate(S_names):
    print(f"Buoy {name} at lat: {S_lat[i]}, lon: {S_lon[i]}")

np.random.seed(42)
permutation = np.random.permutation(X_raw.shape[0])
X_raw = X_raw[permutation]
S_raw = S_raw[permutation]

X_raw = X_raw.reshape(-1, 180 * 360)
S_raw = S_raw.reshape(-1, 20)

print(f"\nReshaped X_raw to: {X_raw.shape}")
print(f"Reshaped S_raw to: {S_raw.shape}")

Loaded SST data with shape: (26, 365, 64800)
Loaded Buoy data with shape: (26, 365, 20)

Buoy 46035 at lat: 57.034, lon: -177.468
Buoy 46001 at lat: 56.296, lon: -148.027
Buoy 46066 at lat: 52.776, lon: -154.992
Buoy 46006 at lat: 40.73, lon: -137.42
Buoy 46059 at lat: 38.067, lon: -129.895
Buoy 46025 at lat: 33.765, lon: -119.077
Buoy 51001 at lat: 24.475, lon: -162.03
Buoy 51004 at lat: 17.504, lon: -152.197
Buoy 52200 at lat: 13.354, lon: 144.788
Buoy 44013 at lat: 42.346, lon: -70.651
Buoy 41001 at lat: 34.738, lon: -72.503
Buoy 41010 at lat: 28.86, lon: -78.478
Buoy 41046 at lat: 23.84, lon: -68.34
Buoy 41043 at lat: 21.09, lon: -64.864
Buoy 41044 at lat: 21.582, lon: -58.63
Buoy 42057 at lat: 16.975, lon: -81.578
Buoy 42058 at lat: 14.114, lon: -75.949
Buoy 42040 at lat: 29.207, lon: -88.237
Buoy 42001 at lat: 25.922, lon: -89.638
Buoy 42002 at lat: 25.95, lon: -93.78

Reshaped X_raw to: (9490, 64800)
Reshaped S_raw to: (9490, 20)


In [30]:
import numpy as np
import matplotlib.pyplot as plt

# Load SST data -> X CLEAN
mean_X = np.mean(X_raw, axis=0)
sst_locs = np.where(mean_X != 0)[0]

X_clean = X_raw[:, sst_locs]
S_clean = S_raw

In [31]:
ERRORS = []

# DataManager
LAGS = 40

TRAIN_SIZE = 0.70        # percentage of data used for training 
VAL_SIZE = 0.15          # percentage of data used for validation
TEST_SIZE = 0.15         # percentage of data used for testing

COMPRESS_SST = 500

# Training
OPTIMIZER = "AdamW"     # optimizer to use
LEARNING_RATE = 5e-4    # learning rate for training
NUM_EPOCHS = 200        # maximum number of epochs for training
LR_SCHEDULER = "CosineAnnealingLR"      # learning rate scheduler to use
SCHEDULER_STEP_SIZE = NUM_EPOCHS        # step size for learning rate scheduler (if applicable)

BATCH_SIZE = 32         # batch size for training
PATIENCE = 80           # number of epochs with no improvement on validation loss before stopping training
VERBOSE_TRAIN = True

# Architecture
ENCODER_DEPTH = 1               # number of encoder layers
HIDDEN_SIZE = 32                # hidden size for encoder and decoder layers
N_HEADS = 4                     # number of attention heads (if using a transformer-based architecture)
DIM_FEEDFORWARD = 48            # dimension of feedforward network in transformer (if using a transformer-based architecture)
DECODER = "cnn"                 # type of decoder to use (mpl or cnn)   
DECODER_HIDDEN_SIZES = [256]    # hidden sizes for decoder layers 
DROPOUT = 0.1                   # dropout rate for encoder and decoder layers
MASKING_TYPE = "mask_data"      # type of masking to apply to inputs during training (mask_data, mask_data_random, etc.)

## Training

In [32]:
# Data manager
manager = DataManager(lags=LAGS, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE)
# add data to manager
manager.add_data(X_clean, id="Surface elevation", compress=COMPRESS_SST, measurements=S_clean)
# prepare datasets
train_dataset, val_dataset, test_dataset = manager.prepare(verbose=True, masking_type=MASKING_TYPE)

# network input and output shapes
nn_input_shape = train_dataset.X.shape[-1]
nn_output_shape = train_dataset.Y.shape[-1]
print("\nInput shape for the model:", nn_input_shape)
print("Output shape for the model:", nn_output_shape)


Datasets: ['Surface elevation']

Number of nan values in sensor measurements after masking: 51379

Data shapes after preparation:
Train:	 torch.Size([6643, 40, 20]) -> torch.Size([6643, 500])
Val:	 torch.Size([1423, 40, 20]) -> torch.Size([1423, 500])
Test:	 torch.Size([1424, 40, 20]) -> torch.Size([1424, 500])

Input shape for the model: 20
Output shape for the model: 500


In [34]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer_learned_mask", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels=True,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT, 
 )
mshred = mSHRED(args=model_args).to(device)

model_path = Path("models_saved/T3G.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)
model_exists = model_path.exists()

#if not already trained, train the model and save it, otherwise load the model from disk
train_time_minutes = None
if not model_exists:
    print("Training mSHRED model...")
    train_start = time.perf_counter()

    # Use cosine annealing LR schedule
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        batch_size=BATCH_SIZE, optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS,
        patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
        masking_type=MASKING_TYPE,
     )

    train_time_minutes = (time.perf_counter() - train_start) / 60
    print(f"Training time: {train_time_minutes:.2f} min")
    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(
    manager, mshred, train_dataset, val_dataset, test_dataset, X_clean,
    model_name="T3G", train_time_minutes=train_time_minutes,
)

del model_args, model_path, mshred
del model_exists
del train_dataset, val_dataset, test_dataset


Loading pre-trained mSHRED model...
